# Engenharia de features -- Continua

Dia 4 (parte 1): transforma `data/processed/municipio_mes.parquet` (uma
linha por município x mês) num dataset supervisionado -- uma linha por
"(município, mês t)", com features calculadas usando só o que já era
conhecido até o mês t, e alvo = `fec_aprox` observado em t+1. A lógica mora
em `ml/features.py` (reaproveitada por este notebook, por
`04-modelagem.ipynb` e por `ml/train.py`) -- este notebook existe para
VALIDAR e justificar essas escolhas com números reais, não para reimplementar
a lógica.

Ver o docstring de `ml/features.py` para a justificativa completa de cada
feature -- resumo:

- Sazonalidade entra via `mes_sin`/`mes_cos` (cíclica), não via uma defasagem
  individual de 12 meses -- com só 24 meses de histórico, essa defasagem
  sazonal não teria quase nenhum dado de TREINO disponível antes da janela
  de teste (ver seção 4).
- Persistência de curto prazo entra via `lag_1`/`lag_2`/`lag_3` e médias
  móveis.
- Causa entra agregada e grosseira (proporção genérica vs. ambiental), não
  como as ~46 colunas originais -- ~95% dos eventos só têm causa genérica
  (achado do Dia 3).


In [1]:
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, r"/home/claude/radar-continuidade")
from ml.features import FEATURES_NUMERICAS, construir_dataset, divisao_temporal, filtrar_utilizaveis

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

ARTIFACTS = r"/home/claude/radar-continuidade/ml/artifacts"
df = pd.read_parquet(r"/home/claude/radar-continuidade/data/processed/municipio_mes.parquet")
dataset = construir_dataset(df)
dataset.shape


(131673, 22)

## 1. Um bug real pego construindo este dataset

A primeira versão de `construir_dataset` calculava `lag_1`/`lag_2`/... por
POSIÇÃO dentro de cada município (`.shift(1)`, `.shift(2)`...), sem levar em
conta se os meses eram realmente consecutivos no calendário. Isso escondia
dois problemas -- só descobertos comparando os números deste notebook
contra o baseline do Dia 3 (que usa um pivô largo por mês-calendário, não
por posição):

1. **Off-by-one:** como o alvo é o mês t+1, "1 mês antes do alvo" é o
   próprio mês t -- não t-1. Uma primeira versão usava `shift(1)` para
   `lag_1`, que na verdade apontava 2 meses antes do alvo.
2. **Buracos no meio do histórico:** município com um mês faltando no meio
   (não no início/fim) fazia o `.shift()` por posição "pular" o buraco e
   tratar dois meses não-consecutivos como vizinhos.

O segundo problema não é hipotético -- é real neste dado:


In [2]:
completo = df.dropna(subset=["nome_municipio"]).copy()
completo["periodo"] = pd.to_datetime(
    {"year": completo["ano"], "month": completo["mes"], "day": 1}
).dt.to_period("M")


def tem_buraco_no_meio(serie_periodos):
    ordenado = serie_periodos.sort_values()
    esperado = pd.period_range(ordenado.min(), ordenado.max(), freq="M")
    return len(esperado) != len(ordenado)


tem_buraco = completo.groupby("codigo_ibge_resolvido")["periodo"].apply(tem_buraco_no_meio)
print(f"{tem_buraco.sum()} de {len(tem_buraco)} municipios "
      f"({tem_buraco.mean():.1%}) tem pelo menos um mes faltando NO MEIO "
      "do historico (nao so no inicio/fim) -- longe de ser caso de borda.")


543 de 5509 municipios (9.9%) tem pelo menos um mes faltando NO MEIO do historico (nao so no inicio/fim) -- longe de ser caso de borda.


`construir_dataset` corrige os dois problemas reindexando cada município
para o calendário completo do painel antes de calcular qualquer defasagem
(ver o docstring da função) -- os testes em `tests/test_ml_features.py`
fixam esse comportamento (inclusive um teste que constrói um município com
um buraco no meio de propósito, para não regredir).

## 2. O dataset supervisionado resultante

In [3]:
print(f"{len(dataset):,} linhas (uma por municipio x mes com nome resolvido)")
dataset[["codigo_ibge_resolvido", "ano", "mes", "lag_1", "lag_2", "alvo", "exposicao"]].head(8)


131,673 linhas (uma por municipio x mes com nome resolvido)


,codigo_ibge_resolvido,ano,mes,lag_1,lag_2,alvo,exposicao
0,1100015,2024,1,0.019718,NaN,0.016658,4575.666667
1,1100015,2024,2,0.016658,0.019718,0.017479,4559.166667
2,1100015,2024,3,0.017479,0.016658,0.013008,4513.333333
3,1100015,2024,4,0.013008,0.017479,0.013502,4702.166667
4,1100015,2024,5,0.013502,0.013008,0.011648,4694.666667
5,1100015,2024,6,0.011648,0.013502,0.013699,4655.166667
6,1100015,2024,7,0.013699,0.011648,0.016777,4639.555556
7,1100015,2024,8,0.016777,0.013699,0.020999,4659.166667


In [4]:
utilizaveis = filtrar_utilizaveis(dataset)
print(f"{len(utilizaveis):,} linhas utilizaveis (tem lag_1 e alvo -- so o ultimo mes de cada "
      f"municipio fica de fora, por nao ter 'mes seguinte' observado ainda)")

nulos = utilizaveis[FEATURES_NUMERICAS].isna().mean().sort_values(ascending=False)
nulos = nulos[nulos > 0]
print("\n% de nulos por feature (esperado para defasagens mais longas no "
      "inicio do historico de cada municipio -- HistGradientBoostingRegressor "
      "lida nativamente, GLM usa mediana do treino, ver ml/modelos.py):")
nulos


125,621 linhas utilizaveis (tem lag_1 e alvo -- so o ultimo mes de cada municipio fica de fora, por nao ter 'mes seguinte' observado ainda)

% de nulos por feature (esperado para defasagens mais longas no inicio do historico de cada municipio -- HistGradientBoostingRegressor lida nativamente, GLM usa mediana do treino, ver ml/modelos.py):


media_movel_3    0.095525
lag_3            0.091617
lag_2            0.047763
dtype: float64

## 3. Corte temporal treino/teste

In [5]:
treino, teste = divisao_temporal(dataset, ano_corte=2025)
print(f"treino: {len(treino):,} linhas, alvo em {(treino['periodo']+1).min()} a {(treino['periodo']+1).max()}")
print(f"teste:  {len(teste):,} linhas, alvo em {(teste['periodo']+1).min()} a {(teste['periodo']+1).max()}")
assert (treino["periodo"] + 1).dt.year.max() < 2025
assert (teste["periodo"] + 1).dt.year.eq(2025).all()
print("\nOK: nenhum mes de 2025 vaza para o treino.")


treino: 60,526 linhas, alvo em 2024-02 a 2024-12
teste:  65,095 linhas, alvo em 2025-01 a 2025-12

OK: nenhum mes de 2025 vaza para o treino.


**Limitação registrada (importante para o Dia 4 e para a conversa sobre
incluir mais anos de dados):** com só 24 meses de histórico (2024+2025), o
treino cobre só 10 meses-alvo (mar/2024 a dez/2024) -- uma defasagem sazonal
individual por município (mesmo mês do ano anterior) só existiria a partir
de jan/2025, que já é o próprio período de teste. Por isso a sazonalidade
entra via `mes_sin`/`mes_cos` (funciona com qualquer profundidade de
histórico, porque aprende o padrão a partir da variação ENTRE municípios no
mesmo mês, não da história de um município individual) em vez de uma
defasagem de 12 meses -- ver `ml/features.py`.

## 4. Poder preditivo bruto de cada feature

In [6]:
correlacoes = utilizaveis[FEATURES_NUMERICAS + ["alvo"]].corr()["alvo"].drop("alvo")
correlacoes = correlacoes.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5.5))
cores = ["firebrick" if v < 0 else "steelblue" for v in correlacoes.values]
ax.barh(correlacoes.index[::-1], correlacoes.values[::-1], color=cores[::-1])
ax.set_xlabel("Correlacao com o alvo (fec_aprox do mes seguinte)")
ax.set_title("Correlacao bruta de cada feature numerica com o alvo")
fig.tight_layout()
fig.savefig(f"{ARTIFACTS}/correlacao_features_alvo.png", dpi=110)
plt.show()

correlacoes.round(3)


lag_1                         0.842
media_movel_3                 0.796
media_movel_expandida         0.783
lag_2                         0.744
lag_3                         0.646
log_consumidores             -0.266
log_n_eventos_lag_1           0.236
mes_sin                      -0.184
mes_cos                       0.169
prop_causa_generica_lag_1     0.146
prop_causa_ambiental_lag_1   -0.108
n_distribuidoras_lag_1       -0.011
Name: alvo, dtype: float64

**Leitura:** `lag_1` (o valor do proprio mes t) e de longe a feature mais
correlacionada com o alvo, seguida pelas outras defasagens/medias moveis --
consistente com o achado do Dia 3 de que o risco e persistente mes a mes.
`mes_sin`/`mes_cos` tem correlacao bem mais fraca ISOLADAMENTE (correlacao
linear simples nao captura bem um efeito ciclico -- o modelo de arvore do
Dia 4 consegue usar melhor essa informacao do que uma correlacao de Pearson
sozinha sugere). As features de causa (`prop_causa_*`) tem correlacao quase
nula -- reforca o achado do Dia 3 de que causa e um dado pouco informativo
na pratica.

## Resumo (Dia 4 -- features)

1. Um bug real de off-by-one e de buracos no meio do historico foi
   encontrado e corrigido comparando este pipeline contra o baseline do Dia
   3 -- fixado em testes automatizados (`tests/test_ml_features.py`).
2. O corte temporal treino/teste deixa só ~10 meses de treino -- uma
   limitacao real de ter so 2 anos de dados, que molda a escolha de
   features (sazonalidade ciclica, nao defasagem de 12 meses).
3. `lag_1` domina o poder preditivo bruto; causa contribui muito pouco --
   consistente com a EDA do Dia 3. Ver `04-modelagem.ipynb` para os modelos
   treinados em cima deste dataset.
